In [24]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torchvision
import torchvision.transforms as transforms
import torch
import torch.nn as nn
from torch.nn.functional import relu
import torch.nn.functional as F

In [25]:
train_data = torchvision.datasets.MNIST(root="MNIST/", 
                                        train=True, 
                                        download=True, 
                                        transform= transforms.ToTensor())

validation_data = torchvision.datasets.MNIST(root="MNIST/", 
                                       train=False, 
                                       download=True,
                                       transform= transforms.ToTensor())

train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True)
validation_loader = torch.utils.data.DataLoader(validation_data, batch_size=32, shuffle=False)

In [26]:
device = torch.device('cuda')

sample,_ = train_data[0]
print(sample.shape)

torch.Size([1, 28, 28])


In [27]:
class CNN(nn.Module):

    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(1,32,kernel_size= 3, padding=1, stride=1) # input size: 28x28x1 -> output size: 28x28x32
        self.maxpol1 = nn.MaxPool2d(3, 2) # output size: ((28-3)/2) + 1 = 13x13x32
        self.conv2 = nn.Conv2d(32,64, kernel_size=5, padding=1, stride=1) # output size: ((13 + 2 *1 - 5)/1)+1 = 11x11x64 
        self.maxpol2 = nn.MaxPool2d(5,2) # output size: ((11-5)/2) + 1= 4x4x64

        #Connected layers
        self.fc1 = nn.Linear(4*4*64, 256)
        self.fc2 = nn.Linear(256,64)
        self.output = nn.Linear(64,10)

    def forward(self, x):
        x = relu(self.conv1(x))
        x = self.maxpol1(x)
        x = relu(self.conv2(x))
        x = self.maxpol2(x)
        x = x.view(-1, 4*4*64)
        x = relu(self.fc1(x))
        x = relu(self.fc2(x))
        x = self.output(x)
        return x
        

In [28]:
model = CNN()
EPOCHS = 30
LR = 0.01

In [29]:
def init_weights(model):
    if type(model) == nn.Linear or type(model) == nn.Conv2d:
        torch.nn.init.xavier_uniform_

In [30]:
model.apply(init_weights)
optimizer = torch.optim.SGD(model.parameters(), lr=LR)
criteria = torch.nn.CrossEntropyLoss()

In [31]:
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []
best_model = None

In [32]:
for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    correction_train = 0
    total_train_sample = 0

    for data, target in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = criteria(output, target)
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        preds = output.argmax(dim=1)
        correction_train += (preds == target).sum().item()
        total_train_sample += target.size(0)

    avg_train_loss = total_train_loss / len(train_loader)
    train_accuracy = correction_train / total_train_sample

    train_losses.append(avg_train_loss)
    train_accuracies.append(train_accuracy) 

    model.eval()
    total_val_loss = 0
    correct_val = 0
    total_val_sample = 0

    with torch.no_grad():
        for data, target in validation_loader:
            output = model(data)
            loss = criteria(output, target)
            total_val_loss += loss.item()
            preds = output.argmax(dim=1)
            correct_val += (preds == target).sum().item()
            total_val_sample += target.size(0)

    avg_val_loss = total_val_loss / len(validation_loader)
    val_accuracy = correct_val / total_val_sample

    val_losses.append(avg_train_loss)
    val_accuracies.append(val_accuracy)

    print(
        f"Epoch {epoch+1}/{EPOCHS}: Train Loss = {avg_train_loss:.4f}, Train accuracy = {train_accuracy:.4f}"
        f"Val Loss = {avg_val_loss:.4f}, Val accuracy = {val_accuracy:.4f}"
        )
    
train_accuracy = correction_train/ total_train_sample
val_accuracy = correct_val / total_val_sample
print(f"Final Training Accuracy = {train_accuracy:.4f}")
print(f"Final Validation Accuracy = {val_accuracy:.4f}")
    
    

Epoch 1/30: Train Loss = 1.1705, Train accuracy = 0.6491Val Loss = 0.2079, Val accuracy = 0.9367
Epoch 2/30: Train Loss = 0.1557, Train accuracy = 0.9521Val Loss = 0.1076, Val accuracy = 0.9660
Epoch 3/30: Train Loss = 0.0947, Train accuracy = 0.9707Val Loss = 0.0699, Val accuracy = 0.9777
Epoch 4/30: Train Loss = 0.0735, Train accuracy = 0.9769Val Loss = 0.0450, Val accuracy = 0.9848
Epoch 5/30: Train Loss = 0.0588, Train accuracy = 0.9815Val Loss = 0.0463, Val accuracy = 0.9852
Epoch 6/30: Train Loss = 0.0503, Train accuracy = 0.9842Val Loss = 0.0405, Val accuracy = 0.9869
Epoch 7/30: Train Loss = 0.0431, Train accuracy = 0.9868Val Loss = 0.0348, Val accuracy = 0.9893
Epoch 8/30: Train Loss = 0.0388, Train accuracy = 0.9880Val Loss = 0.0303, Val accuracy = 0.9900
Epoch 9/30: Train Loss = 0.0352, Train accuracy = 0.9888Val Loss = 0.0351, Val accuracy = 0.9888
Epoch 10/30: Train Loss = 0.0321, Train accuracy = 0.9898Val Loss = 0.0334, Val accuracy = 0.9895
Epoch 11/30: Train Loss = 0.0